# Correcting tune 

## Setting it up step by step

To get the tune correctin running we need 

* import the modules involved,

  * the standard ones 
  * ones from accmls workbench

* load the pre-computed or pre-measured quadrupole response
* activate the devices
* instantiate the correction objects: at this stage you can parameterise them to your needs 
* let it run and watch as the tune controller does its jobs: keep the finger's crossed the first time you run it on the real machine


###  Import of modules

In [ ]:
import logging

Activate the warning level below, otherwise you will see a lot of warnings

In [ ]:
logging.basicConfig(level=logging.WARNING)

Standard python modules used here:

* yaml to read the yaml file containing the response data for the different quadrupoles
* json**s** to convert the data to the data model 

In [ ]:
import asyncio
import jsons
import yaml
from pathlib import Path
import pprint

Now all the different symbols of

* accml
* and accml_lib

These are now imported here from the *work bench*. It is a module similar to matplotlib.pyplot: give an interface
which makes interactive work flow easier. 

* So many symbols are all found in the module *accml.work_bench.all* . These are imported as **wba**.
* Custom modules (e.g. particular machine implementations or certain solutions) are imported from custom

    * bessy2 custom module are imported as **b2**
    * what is required from the tune app is imported as **wbt**
    * ophyd async custom is imported as **wbo**
  

In [ ]:
import accml.work_bench as wb
import accml.work_bench.all as wba
import accml.work_bench.lib_.custom.bessyii as b2
import accml.work_bench.app.tune as wbt
import accml.work_bench.custom.ophyd_async as wbo

### The data of the response matrix as measured before

Now let's import the data of the pre measured response matrix. It should be found in this path

from pathlib import Path
data_dir = Path(__name__).absolute().parent.parent.parent

If not there, here some help to find it

In [ ]:
!find  ../../ -name '*.yml';
!pwd

Now lets load the data and convert it to the appropriate data model

In [ ]:
data_dir = Path(__name__).absolute().parent.parent.parent

In [ ]:
with open(data_dir  / "05_reference_data" / "tune_response_from_simulation.yml") as fp:
    d = yaml.load(fp, yaml.SafeLoader)
dm = jsons.load(d, wbt.TuneResponseCollection)

Have a look to the data model: 

For each magnet the response is available. 
* for each plane
* and for each plan the fitted mean or standard variation



you can see them if you uncomment the next line 

In [ ]:
# pprint.pprint(dm)

You need it for a specific quadrupole? Here you go 

In [ ]:
pprint.pprint(dm.get("Q3PD8R"))

### Combining the views and handling the commands : the managers

accml is designed that one has different views of the same object:

* lattice design tools (e.g. pyat) expects quadrupole with K values or to set the frequency to a cavity
* real world devices need power converters for quadrupoles or a master clock to tune the frequency of the cavity and power amplifiers to drive them.

Accml calls that different views: "design" view for the parameters and lattice elements a lattice design uses. "device" view for the real world devices. Now how to combine them.

yp (for _ellow _ages) : provides you 

Now let's load the managers and then discuss them


In [ ]:
yp, lm, ts = b2.bessyii_load_managers()

yp or "yellow pages" gives you names certain bits and pieces belong to, 
e.g. here you can see the names of all quadrupoles. We will use it further below

In [ ]:
yp.get("quadrupoles")[:10]

the liason manager tells **you who speaks to whom**: e.g.

If we want to change the strength of the is magnet: we need to set the current of that particular power converter

In [ ]:
lat_prop = wba.LatticeElementPropertyID("QIT6R", "main_strength")
dev_prop = lm.forward(lat_prop)
dev_prop

the translation service helps you to find out **what** to say

In [ ]:
to = ts.get(wba.ConversionID(lattice_property_id=lat_prop, device_property_id=dev_prop))

This gives us a translation object which we can now poke to find out what we need. 

The K value of this quadrupole is roughly -2.2, which results in a current of roughly 150 A


In [ ]:
to.forward(-2.2)

This is involved but as you will later on, you hardly get into contact with it, it is handled for you!

### Instantiating the devices

Now lets get the devices:

as BESSY II is an EPICS facility the switch between twin and real machine is straight forward. EPICS itself has a flat namespace. All process variables are addressed by a string. 

* So the machine needs no prefix: so setup is then called with ""
* Any twin has now a perfix:
    *  you can pass it, e.g. setting it "my_twin", given that the twin was started this manner
    *  or you pass **None** and let the function guess the prefix

If you started the BESSY II twin on your local computer, they would most probably match


In [ ]:
devices = b2.bessyii_setup(prefix=None)

## Setting up the proccesing objects

Now all the processing objects need to be set up. This normally only needs to be done once
for your installation and can then kept in one module. Here we introduce you to the details
so that you get a feeling what happens under the hood.

Accml and its workbench is very focused on exchanging messages, these are passed around,
adapted to the view and get executed. Various objects help you with this task

So we need
* the controller part:
   *  the controller itself
   *  the oracle or model that lets it forcast what would be the appropriate step to take
   *  the policy to define which step to actually really take
    
* the measurement execution engine
    * which actually takes care of taking over the commands that actually should be performed
    * probing the detectors it should read
      
* The measurement engine uses a backend to actually talk to the real world devices. Its still under investigation if this layer is needed.

   * When talking to the simulation engine, it uses directly the backend
   * When talking to real world devices, it uses a backend that can talk to ophyd async devices
   * Other backends could be implementd if needed


That seems to be involved, but don't forget: you only need to do it once, unless you want fine grained control.
We walk you through the details so that you get an idea


### The backend

Whe have retrieved the devices above. To be precise these are ophyd async devices.

But as we have to map between the views, its convinient to have a backend

In [ ]:
backend  = wbo.OphydAsyncDeviceBackendRW(devices=devices)

we can ask it which view it understands (the measurement execution engine uses the information so that it knows how to talk to it)

Currently at this stage the connection has to be made active by hand.
We would be interested to here if it always should connect automatically at the first request,  at least for the ophyd async device
that needs it 

In [ ]:
dev = devices.get("Q1PDR")
await dev.connect()

Now its status can be read: it always needs to be triggered first and then it is read.

In [ ]:
await backend.trigger("Q1PDR", "set_current")
await backend.read("Q1PDR", "set_current")

Later on when we do measurements, we typically only measure around the current status.
so here we need a state cache, which works together with the delta backend:

Note: we are currently investigating if the measurement engine should handle the delta directly together with the state cache.
Then accml can persumably blend better in existant measurement orechstration stacks like bluesky and persumably bliss.

But for the time being with have a delta backend. It consists of 
 * the state cache
 * and the backend itself

So let's generate them then, we will inspect them

In [ ]:
state_cache =wba.StateCache(name="BESSY2_OphydAsync_Dev_State_Cache")

In [ ]:
delta_backend = wbo.OphydAsyncDeltaBackendRWProxy(backend=backend, cache=state_cache)

So lets use them: before we asked for the current of the Quadpule:
lets aks the delta backend the same question:

In [ ]:
await delta_backend.trigger("Q1PDR", "set_current")
await delta_backend.read("Q1PDR", "set_current")

How we find the input. Liason manager comes in handy here

In [ ]:
## Todo add a emethod to retrieve all lattice elements and device elemeent properties

In [ ]:
lm

You see that you get the same answer.

Now lets pose it the following question

In [ ]:
await delta_backend.trigger("Q1PDR", "delta_set_current")
await delta_backend.read("Q1PDR", "delta_set_current")

# currenty fails needs to be fixed

And we see it returns 0.0. The state cache, however, now stores the reference value

In [ ]:
await delta_backend.trigger("Q1PDR", "set_current")
await delta_backend.read("Q1PDR", "set_current")

In [ ]:
state_cache.keys()

In [ ]:
state_cache.get(wba.ReadCommand("Q1PDR", property="set_current"))

as you can see it returns a "ReadCommand". We see later what this is. All for now: we reuse it here
as the state cache needs to store the reference value by name and property. 

Now why ReadCommand is not used directly for the backend? accml follows the design choice as 
e.g. taken by the GNU Scientific library: only impose types at each state that bring some real extra to the table

If we now ask the value without delta we get 

In [ ]:
await delta_backend.trigger("Q1PDR", "set_current")
await delta_backend.read("Q1PDR", "set_current")

we could ask the same value to the backend and it would answer

In [ ]:
await backend.trigger("Q1PDR", "set_current")
await backend.read("Q1PDR", "set_current")

The backend does not understand the delta commans

Now changing the value of the quadrupole goes like that:

In [ ]:
await delta_backend.set("Q1PDR", "delta_set_current", 0.1)

All data is stored in a simple storage
Please note that the basic measurement execution engine 

In [ ]:
storage = wba.SimpleDataStorage()

The command rewriter is defined by 

In [ ]:
rewriter = wba.CommandRewriter(liaison_manager=lm, translation_service=ts)

In [ ]:
mexec = wba.BasicMeasurementExecutionEngine(
    backend=delta_backend,
    cmd_rewriter=rewriter,
    storage=storage,
    expected_view_for_output="device",
    num_readings=2,
)

In [ ]:
## Setting up the tune controller

The oracle needs to know what our target is, furthermore it needs the response.

Look it up in the beginning, there we imported the quadrupole responses as a data model

In [ ]:
tune_target = wba.Tune(x=1065, y=855)

So all here to set up the tune oracle .. Internally it just uses a fit for the forecast

In [ ]:
oracle = wbt.TuneOracle(col=dm, target=tune_target)

Now wee want a policy. The oracle says what we would do, but here we state how much we actually do.
Have a look to its interface what it could do. This one is pretty simple it just does what the 
oracle suggested

In [ ]:
policy = wbt.TunePolicy(scale=1.0)

Finally the controller. It uses the measurement execution engine to perform its job.
Furthermore as we can se it comes with some parameters to tune it.

The specific tune device here -- the BESSY II ophyd async tune device (proxy) is implemented in such a way
that it will only return if new data have been received. Still these data arrive too swiftly.

So extra parameters are awailable:

* n_samples: how often to sample the readback, here the tune
* wait between samples: how long to wait between samples
* wait after set: when the controller sent a command it waits this time before requesting new data

N_samples will be typically 1 and the waits will be 0 in the end. In the beginning it is quite 
convienient to be able to tune the controller from here. Try a bit and see if it works!

When it does, tune it stepy by step. 
But we assume that the final production controller on your site will be implemented 
using the interface of your control system or your fast data interfacae without any extra overhead.

This one shall get you started or be useable for tunability.

In [ ]:
controller = wbt.TuneCorrectionController(
    oracle=oracle,
    policy=policy,
    mexec=mexec,
    n_samples=3,
    wait_after_set=1.2,
    wait_between_samples=0.5,
)

## Finally: commands the structured accml way to communication amd measurement organsiation 

In [ ]:
rcmds = [wba.ReadCommand(id="tune", property="transversal")]
set_cmds = [
        wba.ReadCommand(id=elm.pc_name, property="delta_set_current") for elm in dm.col
]

In [ ]:
await asyncio.gather(*[devices.get(rc.id).connect() for rc in rcmds])

In [ ]:
await asyncio.gather(*[devices.get(rc.id).connect() for rc in set_cmds]);

In [ ]:
await controller.continuous(read_commands=rcmds, set_commands=set_cmds, n_steps=2)